# 07. Expert Parallelism and 3D 4D Parallel Composition

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Lepikhin et al. (GShard, 2020), Fedus et al. (Switch Transformer, 2021), Shoeybi et al. (Megatron 3D Parallelism, 2019), DeepSeek-AI (DeepSeek-V3, 2024).
- **Purpose:** Partition Mixture of Experts (MoE) across devices via All-to-All dispatch/combine and hierarchically compose DP, TP, PP, and EP onto physical network topologies.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Pipeline Parallelism GPipe 1F1B Zero Bubble and DualPipe](06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.ipynb) $\to$ `🔗 Downstream Syntheses:` [DiLoCo Local SGD PowerSGD and Distributed Checkpointing](08_diloco_local_sgd_powersgd_and_distributed_checkpointing.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use Expert Parallelism when token batch sizes are too small to saturate local expert GEMMs or when inter-accelerator bisection bandwidth is severely constrained. In high-latency networks, All-to-All communication latency dominates over the compute savings of sparse routing.

## Mathematical Foundations & Governing Equations

$$
y_t = \sum_{j \in \text{Top-}k} p_j \cdot \text{Expert}_j(x_t), \quad R = d \cdot (PP \cdot TP) + p \cdot TP + t
$$

### Asymptotic Time & Memory Complexity
- **Sparse Compute Advantage:** Slashes active FLOPs per token by $\frac{E}{k}\times$ (e.g. $32\times$ for $E=64, k=2$) while maintaining the representational capacity of all $E$ experts.
- **Cartesian Topology Alignment:** Maps $TP$ to intra-node NVLink, $EP$ to intra-rack optical, and $DP/PP$ to inter-rack fabrics.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def compute_3d_mesh_coordinates(dp: int, pp: int, tp: int) -> torch.Tensor:
    world_size = dp * pp * tp
    coords = torch.zeros(world_size, 3, dtype=torch.int64)
    for r in range(world_size):
        dp_idx = r // (pp * tp)
        rem = r % (pp * tp)
        coords[r] = torch.tensor([dp_idx, rem // tp, rem % tp], dtype=torch.int64)
    return coords

def expert_parallel_forward(x: torch.Tensor, w_gate: torch.Tensor, w_experts: torch.Tensor, ep_world_size: int = 2) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    num_tokens, d_model = x.shape
    num_experts = w_experts.shape[0]
    experts_per_rank = num_experts // ep_world_size
    gate_logits = x @ w_gate
    top2_logits, top2_indices = torch.topk(gate_logits, k=2, dim=-1)
    top2_probs = F.softmax(top2_logits, dim=-1)
    dispatch_mask = torch.zeros(ep_world_size, num_tokens, 2, dtype=torch.float32)
    for r in range(ep_world_size):
        r_start, r_end = r * experts_per_rank, (r + 1) * experts_per_rank
        dispatch_mask[r] = ((top2_indices >= r_start) & (top2_indices < r_end)).float()
    moe_out = torch.zeros_like(x)
    for t in range(num_tokens):
        for k in range(2):
            exp_idx = int(top2_indices[t, k])
            moe_out[t] += top2_probs[t, k] * (x[t] @ w_experts[exp_idx])
    return moe_out, top2_indices, top2_probs, dispatch_mask

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

coords = compute_3d_mesh_coordinates(2, 2, 2)
assert float(torch.max(torch.abs(coords - weights["mesh_coords"]))) < 1e-5

moe_out, top2_indices, top2_probs, dispatch_mask = expert_parallel_forward(weights["x"], weights["w_gate"], weights["w_experts"], 2)
assert float(torch.max(torch.abs(moe_out - weights["moe_out"]))) < 1e-5
assert float(torch.max(torch.abs(top2_indices - weights["top2_indices"]))) < 1e-5
assert float(torch.max(torch.abs(top2_probs - weights["top2_probs"]))) < 1e-5
assert float(torch.max(torch.abs(dispatch_mask - weights["dispatch_mask"]))) < 1e-5
print("All Expert Parallelism and 3D Mesh Composition parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: compute_3d_mesh_coordinates(2, 2, 2), iters=50, name="3D Mesh Coordinates"),
    benchmark_fn(lambda: expert_parallel_forward(weights["x"], weights["w_gate"], weights["w_experts"], 2), iters=50, name="Expert Parallel Forward"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Dense FFN FLOP Explosion vs Sparse Expert Parallelism
num_experts = 64
top_k = 2
d_model = 4096
d_ffn = 16384
num_tokens = 2048

# Dense model evaluating all 64 feedforward networks per token:
dense_flops = num_tokens * (2 * num_experts * 2 * d_model * d_ffn)  # up & down projections

# Sparse MoE evaluating only top_k experts per token:
moe_flops = num_tokens * (2 * top_k * 2 * d_model * d_ffn)

print(f"Dense Full-Network FLOPs:  {dense_flops / 1e12:.2f} TFLOPs")
print(f"Sparse Top-2 MoE FLOPs:    {moe_flops / 1e12:.2f} TFLOPs  ({dense_flops / moe_flops:.1f}x compute reduction)")
assert dense_flops == (num_experts / top_k) * moe_flops, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Conditional Computation Scaling:** Decoupling parameter count from per-token compute unlocks trillion-parameter scale at manageable training FLOP budgets.
2. **All-to-All Dispatch Efficiency:** Expert Parallelism relies on high-speed all-to-all crossbars to route tokens to expert-hosting ranks without parameter duplication.
3. **Cartesian Hierarchy Alignment:** Mapping TP to intra-node NVLink, EP to intra-rack optical fabrics, and DP/PP to inter-rack switches matches communication volume to bandwidth.

---
### Cross-Domain Generalisation Matrix
| Domain | Conditional Routing / Mesh Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Distributed Content Delivery (CDN)** | Anycast routing to geographically distributed edge nodes | Serves client requests from nearest cached edge without central backhaul |
| **Hospital Specialty Referral** | Triage routing patient cases to specialized medical clinics | Optimizes specialist doctor utilization while bounding queue wait times |
| **Supercomputer Task Scheduling** | Priority task dispatch to heterogeneous accelerator partitions | Directs compute-bound kernels to GPUs and I/O tasks to NVMe nodes |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_expert_parallelism_and_3d_4d_parallel_composition.md](../../markdown/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.md)
- **PyTorch 2.x Module:** [07_expert_parallelism_and_3d_4d_parallel_composition.py](../../pytorch2/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.py)
- **Functional JAX Module:** [07_expert_parallelism_and_3d_4d_parallel_composition.py](../../jax/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.py)
- **Native Rust Candle Track:** [07_expert_parallelism_and_3d_4d_parallel_composition.rs](../../candle/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.rs)
- **Hardware Kernel (CPU / WGSL):** [07_expert_parallelism_and_3d_4d_parallel_composition.rs](../../kernels/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.rs)
- **Golden SafeTensors:** [07_expert_parallelism_and_3d_4d_parallel_composition.safetensors](../../weights/16_distributed_and_scalable/07_expert_parallelism_and_3d_4d_parallel_composition.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Pipeline Parallelism GPipe 1F1B Zero Bubble and DualPipe](06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 08. DiLoCo Local SGD PowerSGD and Distributed Checkpointing](08_diloco_local_sgd_powersgd_and_distributed_checkpointing.ipynb) &rarr;